<a href="https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Ranked actions + reason codes

**`final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)`** — the same blend formula the starter pipeline uses (`docs/ml-intern-dataset-and-lane-guide.md`, section 5), applied to my own warehouse-scale model (`w05_model`, ML-08) and frozen baseline (`w04_baseline_score`, ML-07) instead of the starter CSV's.

My reason codes are **not** a copy of the starter CSV's set — those rely on `ctr`, `sessions_90d`, `engagement_rate`, and `scroll_rate`, none of which my `w03_data_contract` (ML-04) notebook verified exist on the real warehouse tables I actually queried (`dim_content`, `dim_clients`, `fact_content_daily_performance`). Per that notebook's own rule — *"I haven't run a DESCRIBE on it yet, so I don't know what else it ships... it stays out until I do"* — I only use fields I've verified: `imp_90d`, `pos_90d`, `trend_ratio_90d`, `content_age_days`, `ai_referral_share_90d`, and the model probability.

- `MODEL_DECLINE_RISK` — model probability ≥ 0.65
- `VISIBLE_MODEL_OPPORTUNITY` — model probability ≥ 0.50 and `imp_90d` ≥ 500
- `POSITION_UPSIDE` — `pos_90d` > 10 (off page one) and `imp_90d` ≥ 500 (real demand already exists; a position gain here is realistic, unlike inventing demand from nothing)
- `AI_REFERRAL_GAP` — `ai_referral_share_90d` below the portfolio median while `imp_90d` ≥ 500 (visible pages that AI tools aren't citing yet — the direction the lane guide names as "AI Referral Opportunity")
- `STALE_VISIBLE_STABLE` / `STALE_VISIBLE_DECLINING` — carried over unchanged from the frozen `w04_baseline_score` rule

**Action mapping:** `refresh_now` (decline risk + stale+visible), `refresh_and_expand` (decline risk + position upside), `monitor_ai_opportunity` (AI referral gap, no decline risk), `monitor` (stale+visible+stable only), `no_action` (none of the above).

**Confidence label**, matching the starter's own "enough evidence" idea: `high` needs `final_refresh_score` above its own 80th percentile AND `imp_90d` ≥ 500 AND model probability ≥ 0.50; `medium` needs at least one of those; `low` otherwise.


In [1]:
%pip -q install duckdb

import os, getpass
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

FEATURE_START = "DATE '2026-01-01'"
FEATURE_END   = "DATE '2026-03-31'"
LABEL_START   = "DATE '2026-04-01'"
LABEL_END     = "DATE '2026-04-30'"

frame = con.sql(f"""
    WITH eligible_clients AS (
        SELECT client_hash_id FROM {TABLES['dim_clients']} WHERE gsc_data_start <= {FEATURE_START}
    ),
    feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) AS imp_90d,
               AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS pos_90d,
               SUM(CASE WHEN f.report_date >  {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_first60,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS days_with_impressions_90d,
               SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.sessions_ai ELSE 0 END) AS ai_sessions_90d,
               SUM(f.gsc_clicks) AS clicks_90d
        FROM {FACT} f JOIN eligible_clients c USING (client_hash_id)
        WHERE f.report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
        GROUP BY 1, 2 HAVING imp_90d >= 100
    ),
    label AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_label30
        FROM {FACT} WHERE report_date BETWEEN {LABEL_START} AND {LABEL_END} GROUP BY 1, 2
    ),
    content_age AS (
        SELECT content_hash_id, DATE_DIFF('day', CAST(content_created_date AS DATE), {FEATURE_END}) AS content_age_days
        FROM {TABLES['dim_content']}
    )
    SELECT f.*, COALESCE(l.imp_label30, 0) AS imp_label30,
           f.imp_last30 / NULLIF(f.imp_first60 / 2.0, 0) AS trend_ratio_90d,
           f.ai_sessions_90d / NULLIF(f.clicks_90d, 0) AS ai_referral_share_90d,
           CASE WHEN COALESCE(l.imp_label30, 0) < 0.8 * f.imp_last30 THEN 1 ELSE 0 END AS is_declining_next30,
           ca.content_age_days
    FROM feat f LEFT JOIN label l USING (client_hash_id, content_hash_id)
    LEFT JOIN content_age ca USING (content_hash_id)
""").df().dropna(subset=['content_age_days']).copy()

# Frozen baseline (ML-07), unchanged.
STALE_DAYS, VISIBLE_IMP = 180, 500
frame['stale'] = (frame['content_age_days'] >= STALE_DAYS).astype(int)
frame['visible'] = (frame['imp_90d'] >= VISIBLE_IMP).astype(int)
frame['declining_now'] = (frame['trend_ratio_90d'] < 0.8).astype(int)
frame['baseline_score'] = frame['stale'] * frame['visible'] * frame['imp_90d'] * (1 + frame['declining_now'])

# trend_ratio_90d excluded as a MODEL feature (still used for the baseline's declining_now flag
# and reason codes) -- w05_model's own robustness check showed a decision tree's Precision@50
# collapsing from 0.860 to exactly the frozen baseline's 0.340 once trend_ratio_90d was removed.
# It shares imp_last30 with the label's own reference point (is_declining_next30 is defined
# relative to imp_last30), so its apparent importance was a structural coupling with the label,
# not genuine forward-looking signal -- confirmed by the with/without collapse, not just suspected.
FEATURES = ['imp_90d', 'pos_90d', 'days_with_impressions_90d',
            'ai_referral_share_90d', 'content_age_days']
SEED = 42

# Grouped split, same convention as ML-08/ML-09 -- pick the winning method HONESTLY (by its
# held-out Precision@50), not by assuming random forest wins again.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, te_idx = next(gss.split(frame, groups=frame['client_hash_id']))
train_g, test_g = frame.iloc[tr_idx].copy(), frame.iloc[te_idx].copy()

imputer = SimpleImputer(strategy='median').fit(train_g[FEATURES])
X_train, X_test = imputer.transform(train_g[FEATURES]), imputer.transform(test_g[FEATURES])
y_train, y_test = train_g['is_declining_next30'].values, test_g['is_declining_next30'].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

candidates = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'decision_tree':       DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'random_forest':       RandomForestClassifier(n_estimators=300, random_state=SEED),
}
scored = {}
for name, m in candidates.items():
    m.fit(X_train, y_train)
    proba = m.predict_proba(X_test)[:, 1]
    scored[name] = precision_at_k(proba, y_test, 50)

winner_name = max(scored, key=scored.get)
winner = candidates[winner_name]
print("Candidate Precision@50 on the grouped test split:", {k: round(v, 3) for k, v in scored.items()})
print(f"Winning method for this playbook: {winner_name} ({scored[winner_name]:.3f})")

# Score the FULL frame with the winning, already-fitted model (fit only ever touched train_g).
frame['model_probability'] = winner.predict_proba(imputer.transform(frame[FEATURES]))[:, 1]

# normalized_baseline_score: min-max to [0, 1] so it's on the same scale as model_probability
# before the 70/30 blend, matching the starter's own normalization step.
bmin, bmax = frame['baseline_score'].min(), frame['baseline_score'].max()
frame['normalized_baseline_score'] = (frame['baseline_score'] - bmin) / (bmax - bmin) if bmax > bmin else 0.0
frame['final_refresh_score'] = 100 * (0.70 * frame['model_probability'] + 0.30 * frame['normalized_baseline_score'])

ai_median = frame['ai_referral_share_90d'].median()

def reason_code(r):
    if r['model_probability'] >= 0.65:
        return 'MODEL_DECLINE_RISK'
    if r['model_probability'] >= 0.50 and r['imp_90d'] >= 500:
        return 'VISIBLE_MODEL_OPPORTUNITY'
    if r['stale'] and r['visible'] and r['declining_now']:
        return 'STALE_VISIBLE_DECLINING'
    if r['pos_90d'] > 10 and r['imp_90d'] >= 500:
        return 'POSITION_UPSIDE'
    if pd.notna(r['ai_referral_share_90d']) and r['ai_referral_share_90d'] < ai_median and r['imp_90d'] >= 500:
        return 'AI_REFERRAL_GAP'
    if r['stale'] and r['visible']:
        return 'STALE_VISIBLE_STABLE'
    return 'NO_ACTION'

ACTION = {
    'MODEL_DECLINE_RISK': 'refresh_now', 'STALE_VISIBLE_DECLINING': 'refresh_now',
    'VISIBLE_MODEL_OPPORTUNITY': 'refresh_and_expand', 'POSITION_UPSIDE': 'refresh_and_expand',
    'AI_REFERRAL_GAP': 'monitor_ai_opportunity', 'STALE_VISIBLE_STABLE': 'monitor', 'NO_ACTION': 'no_action',
}

frame['reason_code'] = frame.apply(reason_code, axis=1)
frame['action'] = frame['reason_code'].map(ACTION)

p80 = frame['final_refresh_score'].quantile(0.80)
def confidence(r):
    strong = (r['final_refresh_score'] > p80) and (r['imp_90d'] >= 500) and (r['model_probability'] >= 0.50)
    any_signal = (r['imp_90d'] >= 500) or (r['model_probability'] >= 0.50)
    return 'high' if strong else ('medium' if any_signal else 'low')
frame['confidence'] = frame.apply(confidence, axis=1)

print("\nreason_code counts:\n", frame['reason_code'].value_counts())
print("\naction counts:\n", frame['action'].value_counts())
print("\nconfidence counts:\n", frame['confidence'].value_counts())

queue_cols = ['client_hash_id', 'content_hash_id', 'final_refresh_score', 'reason_code', 'action',
              'confidence', 'model_probability', 'imp_90d', 'pos_90d', 'content_age_days', 'trend_ratio_90d']
queue = frame.sort_values('final_refresh_score', ascending=False)[queue_cols].reset_index(drop=True)
queue.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Candidate Precision@50 on the grouped test split: {'logistic_regression': np.float64(0.6), 'decision_tree': np.float64(0.34), 'random_forest': np.float64(0.22)}
Winning method for this playbook: logistic_regression (0.600)

reason_code counts:
 reason_code
VISIBLE_MODEL_OPPORTUNITY    50750
NO_ACTION                    38435
POSITION_UPSIDE               4613
STALE_VISIBLE_STABLE          3821
STALE_VISIBLE_DECLINING       3780
MODEL_DECLINE_RISK            1370
Name: count, dtype: int64

action counts:
 action
refresh_and_expand    55363
no_action             38435
refresh_now            5150
monitor                3821
Name: count, dtype: int64

confidence counts:
 confidence
medium    59898
low       23663
high      19208
Name: count, dtype: int64


,client_hash_id,content_hash_id,final_refresh_score,reason_code,action,confidence,model_probability,imp_90d,pos_90d,content_age_days,trend_ratio_90d
0,client_23a62021009f63c4,content_6c1f18bf7e30996a,69.380337,MODEL_DECLINE_RISK,refresh_now,high,0.991148,10933.0,22.310452,159,8.862394
1,client_2094c6eb080311d5,content_610b0c06170b2b65,68.722977,MODEL_DECLINE_RISK,refresh_now,medium,0.981757,408.0,7.020555,42,406.000000
2,client_e547b89c05043229,content_d258077ef9c3bf77,65.573052,MODEL_DECLINE_RISK,refresh_now,high,0.934757,2380.0,26.752436,256,0.660704
3,client_e547b89c05043229,content_929f6ea9fbf0511a,64.652667,MODEL_DECLINE_RISK,refresh_now,high,0.922904,1677.0,18.009176,256,1.859609
4,client_e547b89c05043229,content_533406a48b105132,64.609586,MODEL_DECLINE_RISK,refresh_now,high,0.922190,1912.0,26.890753,256,2.192982
5,client_9958f0a7ae1df715,content_c6f328445b5040e0,62.422565,MODEL_DECLINE_RISK,refresh_now,medium,0.891751,398.0,22.630020,447,0.543131
6,client_23a62021009f63c4,content_be5859a0f51b37a4,59.349813,MODEL_DECLINE_RISK,refresh_now,high,0.847854,1518.0,14.662271,159,3.580882
7,client_b10cb2997d0c7c86,content_1822fafe5c56b4bb,58.742353,MODEL_DECLINE_RISK,refresh_now,high,0.838695,1145.0,11.478553,224,1.996510
8,client_23a62021009f63c4,content_ec627b05d5cd6e43,58.163053,MODEL_DECLINE_RISK,refresh_now,high,0.830901,2379.0,17.889239,159,0.426313
9,client_23a62021009f63c4,content_f610582c4a37498b,57.241517,MODEL_DECLINE_RISK,refresh_now,high,0.817736,3871.0,29.132422,146,3.463656


## 2. Intended use and limits

**Who uses this:** a FlyRank content reviewer with limited time, deciding which pages from a large inventory to open first. **For what:** ordering their attention — not an automatic publishing or content-editing decision, and not a client-facing score.

**Where it stops being valid:**
- Outside the client population it was built on — clients with `gsc_data_start` after 2026-01-01 were excluded from training entirely (no full Q1 window), so the model has never seen a ramping-up client's pattern.
- Past the April 2026 label horizon — `is_declining_next30` describes "declining in the 30 days after a Q1 2026 window." A page's queue position is not a forecast for October, only for the window this frame was built on; the frame needs rebuilding on fresh dates before reuse.
- For causal decisions — nothing here says a refresh *will* fix a flagged page (`ml-intern-dataset-and-lane-guide.md`, section 6). It says this page's Q1 signals look like the signals of pages that declined in April in this dataset.
- Below the volume floor — `imp_90d >= 100` was required to even enter the frame; anything thinner never got scored at all and isn't "cleared," it's simply absent from this queue.


In [2]:
n_high = (frame['confidence'] == 'high').sum()
print(f"High-confidence rows: {n_high:,} of {len(frame):,} ({n_high/len(frame):.1%}) --")
print("this queue is meant to hand a reviewer a small, inspectable slice, not the whole inventory at once.")

n_excluded_clients = con.sql(f"""
    SELECT COUNT(*) FROM {TABLES['dim_clients']} WHERE gsc_data_start > {FEATURE_START}
""").fetchone()[0]
n_total_clients = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone()[0]
print(f"\nClients excluded from this frame for not having a full Q1 window: {n_excluded_clients} of {n_total_clients} total --")
print("this queue says nothing about those clients' content.")


High-confidence rows: 19,208 of 102,769 (18.7%) --
this queue is meant to hand a reviewer a small, inspectable slice, not the whole inventory at once.

Clients excluded from this frame for not having a full Q1 window: 27 of 104 total --
this queue says nothing about those clients' content.


## 3. Human review + the no-go list

**What a person must check before acting on any `refresh_now` / `refresh_and_expand` row:**
- Read the actual page. A reason code is a hypothesis, not a verdict — check it isn't intentional evergreen content, a recently-launched page still ramping up, or a page tied to a seasonal calendar (`ml-intern-dataset-and-lane-guide.md` section 7's consolidation/seasonality/noise checklist).
- Check for a sibling page. If a related page on the same client absorbed the traffic (consolidation), refreshing the flagged page is the wrong fix — merging or redirecting might be the right one, and this queue can't tell the two apart.
- Confirm the client relationship allows this kind of change before anything goes live — this queue never sees contractual or editorial constraints.

**Never automate:**
- Auto-publishing a refresh from this score alone.
- Auto-messaging a client that their content is "declining" — the label is a 30-day proxy on one dataset slice, not a client-facing diagnosis.
- Treating `low` confidence rows as "safe to ignore" — `low` means the evidence is thin, not that the page is fine; it just means a human needs to look closer before this system can say anything useful either way.


In [3]:
# Concrete no-go cases: rows this queue is least equipped to judge on its own.
edge_low_position_signal = frame[frame['pos_90d'].isna()]
print(f"Rows with no valid position signal in Q1 (pos_90d null -- every gsc_avg_position reading was 0): "
      f"{len(edge_low_position_signal):,}")
print("These pages never showed up ranked anywhere Google reports a position for -- action/confidence still")
print("computed for them from imp_90d and trend alone, but a human should treat the missing position as a")
print("real gap in the evidence, not a neutral zero.")

near_boundary = frame[(frame['final_refresh_score'] > p80 * 0.95) & (frame['final_refresh_score'] < p80 * 1.05)]
print(f"\nRows within 5% of the high-confidence score cutoff: {len(near_boundary):,} -- a small change in the")
print("model or baseline would flip these between medium and high confidence; worth a second look before")
print("treating the cutoff itself as a hard line.")


Rows with no valid position signal in Q1 (pos_90d null -- every gsc_avg_position reading was 0): 0
These pages never showed up ranked anywhere Google reports a position for -- action/confidence still
computed for them from imp_90d and trend alone, but a human should treat the missing position as a
real gap in the evidence, not a neutral zero.

Rows within 5% of the high-confidence score cutoff: 28,194 -- a small change in the
model or baseline would flip these between medium and high confidence; worth a second look before
treating the cutoff itself as a hard line.


## 4. Monitoring / retrain triggers

What would tell me these recommendations went stale:

- **Base-rate drift.** If the April `is_declining_next30` rate (this frame: printed below) moves by more than a few points in a later month's re-run, the whole scoring calibration (in particular the `final_refresh_score` percentile cutoffs) needs rebuilding, not just re-scoring with old weights.
- **Precision@50 drop on a fresh month.** If a later Q-window re-run's grouped-holdout Precision@50 falls toward the frozen baseline's number, the model's edge has eroded and it's time to retrain, not keep shipping the current one.
- **Feature distribution shift.** A meaningful change in `ai_referral_share_90d`'s median (AI referral behavior is new and still moving, per the FlyRank paper's own monthly trend chart) would silently move the `AI_REFERRAL_GAP` reason code's threshold under the queue without anyone changing the code.
- **New clients entering the eligible set.** Each re-run should recheck how many clients now have a full feature window (section 2) — a big jump changes who this queue is even scored for.


In [4]:
current_base_rate = frame['is_declining_next30'].mean()
current_p50_grouped = scored[winner_name]
current_baseline_p50 = precision_at_k(test_g['baseline_score'].values, y_test, 50)
current_ai_median = ai_median

print("Values to compare against on the NEXT scheduled re-run (retrain if any drifts materially):")
print(f"  base rate (is_declining_next30):        {current_base_rate:.3f}")
print(f"  winning model Precision@50 (grouped):    {current_p50_grouped:.3f}  ({winner_name})")
print(f"  frozen baseline Precision@50 (grouped):  {current_baseline_p50:.3f}")
print(f"  ai_referral_share_90d median:            {current_ai_median:.4f}")
print(f"  eligible clients in frame:               {frame['client_hash_id'].nunique()}")


Values to compare against on the NEXT scheduled re-run (retrain if any drifts materially):
  base rate (is_declining_next30):        0.494
  winning model Precision@50 (grouped):    0.600  (logistic_regression)
  frozen baseline Precision@50 (grouped):  0.340
  ai_referral_share_90d median:            0.0000
  eligible clients in frame:               30


## 5. Exports for the paper

Writing the ranked queue and the metrics receipts to `work/outputs/` — per `README.md`'s own rule, these committed JSON/CSV files are what my capstone paper's numbers trace back to, not a claim taken on faith.


In [5]:
import json as _json
import os

os.makedirs('work/outputs', exist_ok=True)

queue.to_csv('work/outputs/action_playbook_queue.csv', index=False)

metrics = {
    'generated_from': 'w07_action_playbook.ipynb (ML-10)',
    'feature_window': ['2026-01-01', '2026-03-31'],
    'label_window': ['2026-04-01', '2026-04-30'],
    'label': 'is_declining_next30',
    'features': FEATURES,
    'split': 'grouped_by_client_hash_id, test_size=0.20, seed=42',
    'n_rows_frame': int(len(frame)),
    'n_clients_frame': int(frame['client_hash_id'].nunique()),
    'base_rate': float(current_base_rate),
    'candidate_precision_at_50': {k: float(v) for k, v in scored.items()},
    'winning_model': winner_name,
    'baseline_precision_at_50': float(current_baseline_p50),
    'reason_code_counts': frame['reason_code'].value_counts().to_dict(),
    'action_counts': frame['action'].value_counts().to_dict(),
    'confidence_counts': frame['confidence'].value_counts().to_dict(),
    'high_confidence_rows': int(n_high),
}

with open('work/outputs/model_results.json', 'w') as f:
    _json.dump(metrics, f, indent=2)

print("wrote work/outputs/action_playbook_queue.csv --", len(queue), "rows")
print("wrote work/outputs/model_results.json")
print("\nCommit both files -- the capstone_report_template.md reproducibility section requires the metrics")
print("file that any sealed/holdout claim traces back to, not just the notebook that produced it once.")


wrote work/outputs/action_playbook_queue.csv -- 102769 rows
wrote work/outputs/model_results.json

Commit both files -- the capstone_report_template.md reproducibility section requires the metrics
file that any sealed/holdout claim traces back to, not just the notebook that produced it once.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
